# ADNI1 vs Healthy MRI Classification (`.nii.gz`)

该版本已从 CT/PNG 流程改为 MRI NIfTI 流程：
- 递归读取两类目录中的 `.nii.gz`（或 `.nii`）
- `ADNI1` 目录标记为阿尔兹海默症（label=1）
- 健康目录标记为正常（label=0）
- 不再进行文件重命名
- 使用 3D CNN 完成训练与评估


In [ ]:
# 如果环境缺少依赖，请取消注释后执行：
# !pip install nibabel scipy scikit-learn tensorflow

import os
import random
from pathlib import Path

os.environ["TF_ENABLE_XLA"] = "0"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/home/biolab_374/anaconda3/envs/dl-assignment --xla_gpu_enable_triton_gemm=false"

# 避免 XLA 触发 libdevice.10.bc 错误（需在 import tensorflow 前设置）
os.environ.setdefault('TF_XLA_FLAGS', '--tf_xla_auto_jit=0')
if 'XLA_FLAGS' not in os.environ and os.environ.get('CONDA_PREFIX'):
    os.environ['XLA_FLAGS'] = f"--xla_gpu_cuda_data_dir={os.environ['CONDA_PREFIX']}"

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# 双保险：禁用 TF XLA JIT
tf.config.optimizer.set_jit(False)
print("TF_ENABLE_XLA =", os.environ.get("TF_ENABLE_XLA"))
print("TF_XLA_FLAGS  =", os.environ.get("TF_XLA_FLAGS"))
print("XLA_FLAGS     =", os.environ.get("XLA_FLAGS"))
tf.config.optimizer.set_jit(False)

# 可选：按需申请显存，降低 OOM 风险
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

print('TensorFlow:', tf.__version__)
print('GPU devices:', gpus)
print('TF_XLA_FLAGS:', os.environ.get('TF_XLA_FLAGS'))
print('XLA_FLAGS:', os.environ.get('XLA_FLAGS'))


In [ ]:
# ----------------------------
# Config
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 修改为你的真实路径
ADNI1_DIR = Path('/home/biolab_374/PycharmProjects/alz_image_classification/BIOCB26/image/ADNI-output')      # Alzheimer (label=1)
HEALTHY_DIR = Path('/home/biolab_374/PycharmProjects/alz_image_classification/BIOCB26/image/Diffusion 7T Preprocessed Recommended')

TARGET_SHAPE = (128, 128, 128)   # (D, H, W)
BATCH_SIZE = 4
EPOCHS = 30
LEARNING_RATE = 1e-4

assert ADNI1_DIR.exists(), f'ADNI1_DIR not found: {ADNI1_DIR}'
assert HEALTHY_DIR.exists(), f'HEALTHY_DIR not found: {HEALTHY_DIR}'


In [ ]:
# ----------------------------
# Recursive scan of NIfTI files
# ----------------------------
def list_nifti_files(root: Path):
    nii_gz = list(root.rglob('*.nii.gz'))
    nii = list(root.rglob('*.nii'))
    # 去重：避免 .nii.gz 被重复匹配
    all_files = sorted(set(nii_gz + [p for p in nii if not str(p).endswith('.nii.gz')]))
    return all_files

adni_files = list_nifti_files(ADNI1_DIR)
healthy_files = list_nifti_files(HEALTHY_DIR)

print('ADNI1 files:', len(adni_files))
print('Healthy files:', len(healthy_files))

records = []
for p in adni_files:
    records.append({'filepath': str(p), 'label': 1, 'group': 'ADNI1'})
for p in healthy_files:
    records.append({'filepath': str(p), 'label': 0, 'group': 'HEALTHY'})

df = pd.DataFrame(records)
print(df['group'].value_counts())
df.head()


In [ ]:
# ----------------------------
# Train/Val/Test split
# ----------------------------
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, random_state=SEED, stratify=train_df['label']
)

print('Train:', len(train_df), train_df['label'].value_counts().to_dict())
print('Val  :', len(val_df), val_df['label'].value_counts().to_dict())
print('Test :', len(test_df), test_df['label'].value_counts().to_dict())


In [ ]:
# ----------------------------
# NIfTI load + preprocess
# ----------------------------
def load_nifti_volume(path: str, target_shape=TARGET_SHAPE):
    img = nib.load(path)
    vol = img.get_fdata(dtype=np.float32)

    # 兼容 4D 数据，默认取第一个时间点/通道
    if vol.ndim == 4:
        vol = vol[..., 0]
    if vol.ndim != 3:
        raise ValueError(f'Unsupported volume shape: {vol.shape} @ {path}')

    # 强度归一化（对非零脑区做 robust min-max）
    mask = vol > 0
    if np.any(mask):
        vox = vol[mask]
        lo, hi = np.percentile(vox, [1, 99])
        vol = np.clip(vol, lo, hi)
        vol = (vol - lo) / (hi - lo + 1e-8)
        vol[~mask] = 0.0
    else:
        vol = np.zeros_like(vol, dtype=np.float32)

    # 重采样到固定形状
    factors = [t / s for t, s in zip(target_shape, vol.shape)]
    vol = zoom(vol, factors, order=1)

    # 添加通道维度: (D,H,W,1)
    vol = vol[..., np.newaxis].astype(np.float32)
    return vol


def tf_load_nifti(path, label):
    def _py_load(p):
        # tf.py_function 传入的是 EagerTensor，不是 Python bytes
        p = p.numpy().decode('utf-8')
        try:
            return load_nifti_volume(p, TARGET_SHAPE)
        except Exception as e:
            raise ValueError(f'Failed to load NIfTI: {p} | {e}')

    vol = tf.py_function(func=_py_load, inp=[path], Tout=tf.float32)
    vol.set_shape(TARGET_SHAPE + (1,))
    label = tf.cast(label, tf.float32)
    return vol, label


def make_dataset(input_df, batch_size=BATCH_SIZE, training=False):
    paths = input_df['filepath'].values
    labels = input_df['label'].values.astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(input_df), seed=SEED)

    ds = ds.map(tf_load_nifti, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)


In [ ]:
# ----------------------------
# 可视化样本（中间切片）
# ----------------------------
sample_path = train_df.iloc[0]['filepath']
sample_label = train_df.iloc[0]['label']
vol = load_nifti_volume(sample_path)
mid_d = vol.shape[0] // 2
mid_h = vol.shape[1] // 2
mid_w = vol.shape[2] // 2

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(vol[mid_d, :, :, 0], cmap='gray')
ax[0].set_title('Axial')
ax[1].imshow(vol[:, mid_h, :, 0], cmap='gray')
ax[1].set_title('Coronal')
ax[2].imshow(vol[:, :, mid_w, 0], cmap='gray')
ax[2].set_title('Sagittal')
for a in ax:
    a.axis('off')
plt.suptitle(f'Label={int(sample_label)} | {Path(sample_path).name}')
plt.tight_layout()
plt.show()


In [ ]:
# ----------------------------
# 3D CNN model
# ----------------------------
def build_3d_cnn(input_shape=TARGET_SHAPE + (1,)):
    model = keras.Sequential([
        keras.layers.Input(shape=input_shape),
        keras.layers.Conv3D(16, 3, padding='same', activation='relu'),
        keras.layers.MaxPool3D(pool_size=2),

        keras.layers.Conv3D(32, 3, padding='same', activation='relu'),
        keras.layers.MaxPool3D(pool_size=2),

        keras.layers.Conv3D(64, 3, padding='same', activation='relu'),
        keras.layers.MaxPool3D(pool_size=2),

        keras.layers.GlobalAveragePooling3D(),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    return model

model = build_3d_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
    jit_compile=False
)
model.summary()


In [ ]:
# ----------------------------
# Train
# ----------------------------
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values
)
class_weight = {0: float(class_weights_arr[0]), 1: float(class_weights_arr[1])}
print('class_weight:', class_weight)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=6, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('mri_3dcnn_best.keras', monitor='val_auc', mode='max', save_best_only=True)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# ----------------------------
# Evaluate
# ----------------------------
test_metrics = model.evaluate(test_ds, verbose=1)
print('Test metrics:', dict(zip(model.metrics_names, test_metrics)))

# 预测与报告
y_true = test_df['label'].values
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, digits=4))
print('Confusion Matrix:', confusion_matrix(y_true, y_pred))


In [ ]:
# ----------------------------
# Plot training curves
# ----------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.plot(history.history['auc'], label='train_auc')
plt.plot(history.history['val_auc'], label='val_auc')
plt.legend()
plt.title('Accuracy / AUC')

plt.tight_layout()
plt.show()


## Notes
- 本 notebook 已去掉原始工程中的批量重命名逻辑。
- 输入数据直接来自递归扫描的 `.nii.gz/.nii`。
- 如果显存不足，可降低 `TARGET_SHAPE`（如 `96,96,96`）和 `BATCH_SIZE`。
- 若要继续使用 EWC，可在本 3D CNN 的训练阶段叠加 EWC 正则项。当前版本先保证 MRI 主流程稳定可用。
